# Modified VIEScore for RefEdit-Bench (Mixed Prompts)

Scores **existing generated RefEdit outputs**; this notebook does not run the editing agent. This notebook scores the latest `mixed` inference run by default; set `MMEDIT_REFEDIT_BENCH_MIXED_RUN` to select a specific run. It loads annotations from the latest `.out/annotation/refedit_bench/run_*/refedit_bench_mixed.jsonl`, ignoring the legacy flat-file override. It reads that inference run's `results/results.jsonl` and `results/images/*_output.png`, so saved images remain discoverable if a subset run replaced the JSONL summary. Scoring uses RefEdit's `metadata.edit_instruction_single`, the dataset GT polygon mask, and identical zero-padding tight source/output crops. Scores are stored separately in that run's `metrics/modified_viescore.jsonl`; inference never creates this directory. 

Install GEdit-Bench VIEScore dependencies in this notebook's Python environment (including `regex` and `requests`), clone [Step1X-Edit](https://github.com/stepfun-ai/Step1X-Edit), and set `VIESCORE_ROOT` to its `GEdit-Bench` directory. Set `VIESCORE_KEY_PATH` to a local file containing only your OpenAI API key on the first line. The scorer requests the `gpt-4o` API model; API usage may incur charges. Never put the key in a cell or commit its file.

In [ ]:
import os
from pathlib import Path

from framework.utils.project import load_project_environment
from evaluation.notebook import load_benchmark, resolve_run_directory

BENCHMARK = "refedit_bench"
PROMPT_MODE = "mixed"
PROJECT_ROOT = load_project_environment()
ANNOTATION_ROOT = Path(
    os.environ.get("ANNOTATE_OUT")
    or PROJECT_ROOT / ".out" / "annotation" / BENCHMARK
).expanduser()
if not ANNOTATION_ROOT.is_absolute():
    ANNOTATION_ROOT = PROJECT_ROOT / ANNOTATION_ROOT
ANNOTATION_ROOT = ANNOTATION_ROOT.resolve()
ANNOTATION_RUNS = sorted(
    path
    for path in ANNOTATION_ROOT.glob("run_*")
    if path.is_dir() and (path / "refedit_bench_mixed.jsonl").is_file()
)
if not ANNOTATION_RUNS:
    raise FileNotFoundError(
        f"No RefEdit annotation run containing refedit_bench_mixed.jsonl "
        f"found under {ANNOTATION_ROOT}."
    )
os.environ["MMEDIT_REFEDIT_BENCH_ANNOTATIONS"] = str(
    ANNOTATION_RUNS[-1] / "refedit_bench_mixed.jsonl"
)
context = load_benchmark(BENCHMARK, prompt_mode=PROMPT_MODE)
ANNOTATION_FILE = context.annotation_file
OUTPUT_ROOT = context.output_dir
RUN_NAME = os.environ.get("MMEDIT_REFEDIT_BENCH_MIXED_RUN") or None
RUN_NAME = "run_20261025_132256_963056"
OUTPUT_DIR = resolve_run_directory(OUTPUT_ROOT, RUN_NAME)
records = context.records
image_loader = context.image_loader
if not OUTPUT_DIR.is_dir():
    raise FileNotFoundError(f"RefEdit output directory not found: {OUTPUT_DIR}")
VIESCORE_ROOT = Path(os.environ.get("VIESCORE_ROOT") or ".").expanduser()
VIESCORE_KEY_PATH = Path(os.environ.get("VIESCORE_KEY_PATH") or ".").expanduser()
print("Annotations:", ANNOTATION_FILE)
print("Selected run:", OUTPUT_DIR)
print("VIEScore checkout:", VIESCORE_ROOT if str(VIESCORE_ROOT) != "." else "not configured")
print("API key file:", VIESCORE_KEY_PATH if str(VIESCORE_KEY_PATH) != "." else "not configured")

Annotations: /Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.out/annotation/refedit_bench/run_20261022_000000_000000/refedit_bench_mixed.jsonl
Selected run: /Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.out/evaluation/refedit_bench/mixed/run_20261025_124646_118000
VIEScore checkout: /Users/nghialt3670/Projects/Step1X-Edit/GEdit-Bench
API key file: /Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.viescore.key


In [9]:
from evaluation.common import HFImageLoader, read_jsonl
from evaluation.refedit_bench_viescore import collect_saved_outputs

records = list(read_jsonl(ANNOTATION_FILE))
if not records:
    raise ValueError(f'No RefEdit annotations found in {ANNOTATION_FILE}')
image_loader = HFImageLoader()
saved_outputs = collect_saved_outputs(
    records,
    OUTPUT_DIR / 'results' / 'results.jsonl',
    OUTPUT_DIR / 'results' / 'images',
    prompt_mode=PROMPT_MODE,
)
print(f'Found {len(saved_outputs)} saved output images:')
for saved in saved_outputs:
    print(f"  [{saved['annotation_index']}] {saved['sample_id']} ({saved['discovered_from']})")

Found 200 saved output images:
  [0] easy:000 (output image)
  [1] easy:001 (output image)
  [2] easy:002 (output image)
  [3] easy:003 (output image)
  [4] easy:004 (output image)
  [5] easy:005 (output image)
  [6] easy:006 (output image)
  [7] easy:007 (output image)
  [8] easy:008 (output image)
  [9] easy:009 (output image)
  [10] easy:010 (output image)
  [11] easy:011 (output image)
  [12] easy:012 (output image)
  [13] easy:013 (output image)
  [14] easy:014 (output image)
  [15] easy:015 (output image)
  [16] easy:016 (output image)
  [17] easy:017 (output image)
  [18] easy:018 (output image)
  [19] easy:019 (output image)
  [20] easy:020 (output image)
  [21] easy:021 (output image)
  [22] easy:022 (output image)
  [23] easy:023 (output image)
  [24] easy:024 (output image)
  [25] easy:025 (output image)
  [26] easy:026 (output image)
  [27] easy:027 (output image)
  [28] easy:028 (output image)
  [29] easy:029 (output image)
  [30] easy:030 (output image)
  [31] easy:031 (o

## Score saved outputs

Run this cell after reviewing the discovered sample list above. Unchanged images reuse cached VIEScore rows; changed or newly discovered images are evaluated and appended to the output JSONL.

## Configure VIEScore

This uses the upstream VIEScore `tie` task with the GPT-4o API model. The API key is read from the local key file; it is never displayed.

## Score saved outputs

The loop below checks annotation/dataset consistency, reuses unchanged cached rows, scores uncached outputs, and atomically rewrites the JSONL after each sample so progress is preserved.

In [10]:
from IPython.display import display
from evaluation.refedit_bench_viescore import evaluate_saved_refedit_bench_outputs

metric_rows, score_summary = evaluate_saved_refedit_bench_outputs(
    records,
    output_dir=OUTPUT_DIR,
    image_loader=image_loader,
    viescore_root=VIESCORE_ROOT,
    key_path=VIESCORE_KEY_PATH,
    prompt_mode=PROMPT_MODE,
)
print(f"Per-sample Modified VIEScore: {OUTPUT_DIR / 'metrics' / 'modified_viescore.jsonl'}")
display([{"category": category, **metrics} for category, metrics in score_summary.items()])

Modified VIEScore: 100%|██████████| 200/200 [00:00<00:00, 1525.59sample/s]

Scored 200 saved RefEdit outputs (100 easy, 100 hard).
Per-sample Modified VIEScore: /Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.out/evaluation/refedit_bench/mixed/run_20261025_124646_118000/metrics/modified_viescore.jsonl
Per-sample Modified VIEScore: /Users/nghialt3670/Projects/MiMo-Edit/mimo-edit/.out/evaluation/refedit_bench/mixed/run_20261025_124646_118000/metrics/modified_viescore.jsonl


[{'category': 'easy',
  'sc': 6.94,
  'pq': 4.99,
  'o': 5.134892847026305,
  'count': 100},
 {'category': 'hard',
  'sc': 6.58,
  'pq': 4.75,
  'o': 4.918346113411034,
  'count': 100}]